# Iterative Iterative Model Development for Fuel Blend Prediction

**Experimentation and Optimization Journey**

This notebook documents the iterative development process of improving fuel blend property prediction models, showcasing different approaches tried and lessons learned.

## 📋 Development Overview
- **Dataset**: Fuel blend composition and component properties
- **Task**: Predict 10 blend properties from 69 input features
- **Goal**: Minimize Mean Absolute Percentage Error (MAPE)
- **Approach**: Systematic experimentation and optimization

## Lab Experimentation Strategy
1. **Baseline Model**: Simple LightGBM with basic features
2. **Feature Engineering**: Progressive addition of domain knowledge
3. **Model Tuning**: Individual target optimization
4. **Advanced Techniques**: Interaction features and alternative approaches
5. **Performance Analysis**: Detailed evaluation and insights

In [ ]:
# Install required packages
!pip install -q lightgbm scikit-learn matplotlib seaborn pandas numpy

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("OK All libraries imported successfully!")

## Data Data Loading and Initial Setup

In [ ]:
# Load datasets
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Data Train shape: {train.shape}")
print(f"Data Test shape: {test.shape}")

# Define column groups
blend_frac_cols = [f"Component{i}_fraction" for i in range(1, 6)]
component_prop_cols = [f"Component{i}_Property{j}" for i in range(1, 6) for j in range(1, 11)]
target_cols = [f"BlendProperty{i}" for i in range(1, 11)]

print(f"Chemistry Blend fractions: {len(blend_frac_cols)} columns")
print(f"Science Component properties: {len(component_prop_cols)} columns")
print(f"Target Target properties: {len(target_cols)} columns")

# Normalize blend fractions (ensure they sum to 1.0)
print("\nChemistry Normalizing blend fractions...")
train[blend_frac_cols] = train[blend_frac_cols].div(train[blend_frac_cols].sum(axis=1), axis=0)
test[blend_frac_cols] = test[blend_frac_cols].div(test[blend_frac_cols].sum(axis=1), axis=0)

print(f"OK Normalization check - Train sums: {train[blend_frac_cols].sum(axis=1).mean():.6f}")
print(f"OK Normalization check - Test sums: {test[blend_frac_cols].sum(axis=1).mean():.6f}")

## Lab Experiment 1: Baseline Model

In [ ]:
# Create basic weighted features
def add_weighted_blend_features(df):
    for prop in range(1, 11):
        weighted_sum = 0
        for comp in range(1, 6):
            frac_col = f"Component{comp}_fraction"
            prop_col = f"Component{comp}_Property{prop}"
            weighted_sum += df[frac_col] * df[prop_col]
        df[f"WeightedProperty{prop}"] = weighted_sum
    return df

# Apply to train and test
train = add_weighted_blend_features(train)
test = add_weighted_blend_features(test)

# Add aggregate features
train["ComponentProp_mean"] = train[component_prop_cols].mean(axis=1)
train["ComponentProp_std"] = train[component_prop_cols].std(axis=1)
test["ComponentProp_mean"] = test[component_prop_cols].mean(axis=1)
test["ComponentProp_std"] = test[component_prop_cols].std(axis=1)

# Define features
weighted_cols = [f"WeightedProperty{j}" for j in range(1, 11)]
aggregate_cols = ["ComponentProp_mean", "ComponentProp_std"]
feature_cols = blend_frac_cols + component_prop_cols + weighted_cols + aggregate_cols

print(f"Tool Total features: {len(feature_cols)}")
print(f"New features: {weighted_cols + aggregate_cols}")

# Split data
X = train[feature_cols]
y = train[target_cols]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Experiment 1: Train individual models for each target
print("Lab Experiment 1: Individual LightGBM models")
print("-" * 50)

models_exp1 = []
val_preds_exp1 = np.zeros_like(y_val)

for i, target in enumerate(target_cols):
    print(f"Training model for {target}...")
    model = lgb.LGBMRegressor(random_state=42, verbose=-1)
    model.fit(X_train, y_train[target])
    val_preds_exp1[:, i] = model.predict(X_val)
    models_exp1.append(model)

# Evaluate
exp1_mape = mean_absolute_percentage_error(y_val, val_preds_exp1)
print(f"\nData Experiment 1 MAPE: {exp1_mape:.6f}")

# Individual target MAPEs
print("\nIndividual target MAPEs:")
for i, target in enumerate(target_cols):
    target_mape = mean_absolute_percentage_error(y_val[target], val_preds_exp1[:, i])
    print(f"{target}: {target_mape:.6f}")

# Identify challenging targets
target_mapes = [mean_absolute_percentage_error(y_val[target], val_preds_exp1[:, i]) for i, target in enumerate(target_cols)]
hard_targets = [target_cols[i] for i, mape in enumerate(target_mapes) if mape > np.mean(target_mapes)]
print(f"\nTarget Challenging targets: {hard_targets}")

## Lab Experiment 2: Targeted Optimization for Hard Targets

In [ ]:
# Experiment 2: Optimize models for challenging targets
print("Lab Experiment 2: Targeted optimization for hard targets")
print("-" * 60)

improved_models = {}
val_preds_exp2 = val_preds_exp1.copy()

for target in hard_targets:
    print(f"\nTool Optimizing {target} with log-transform and tuning...")
    
    # Apply log transformation for stability
    y_train_log = np.log1p(np.maximum(y_train[target], 1e-6))
    y_val_log = np.log1p(np.maximum(y_val[target], 1e-6))
    
    # LightGBM with tuned parameters
    train_set = lgb.Dataset(X_train, label=y_train_log)
    val_set = lgb.Dataset(X_val, label=y_val_log)
    
    params = {
        "objective": "regression",
        "learning_rate": 0.02,
        "num_leaves": 64,
        "max_depth": 7,
        "reg_alpha": 1.0,
        "reg_lambda": 1.0,
        "n_estimators": 2000,
        "metric": "mape",
        "verbosity": -1,
        "seed": 42
    }
    
    # Train with early stopping
    model = lgb.train(
        params,
        train_set,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
    )
    
    # Reverse transform predictions
    val_preds_log = model.predict(X_val, num_iteration=model.best_iteration)
    val_preds_transformed = np.expm1(val_preds_log)
    
    # Update predictions
    target_idx = target_cols.index(target)
    val_preds_exp2[:, target_idx] = val_preds_transformed
    improved_models[target] = model
    
    # Show improvement
    old_mape = mean_absolute_percentage_error(y_val[target], val_preds_exp1[:, target_idx])
    new_mape = mean_absolute_percentage_error(y_val[target], val_preds_transformed)
    improvement = (old_mape - new_mape) / old_mape * 100
    print(f"OK {target}: {old_mape:.6f} → {new_mape:.6f} ({improvement:+.1f}%)")

# Overall Experiment 2 MAPE
exp2_mape = mean_absolute_percentage_error(y_val, val_preds_exp2)
improvement = (exp1_mape - exp2_mape) / exp1_mape * 100
print(f"\nData Experiment 2 MAPE: {exp2_mape:.6f} ({improvement:+.1f}% vs Exp1)")

## Lab Experiment 3: Advanced Feature Engineering

In [ ]:
# Experiment 3: Add interaction features
print("Lab Experiment 3: Adding interaction features")
print("-" * 50)

# Create interaction features (fraction × property)
def add_interaction_features(df):
    for i in range(1, 6):  # Components 1–5
        for j in range(1, 11):  # Properties 1–10
            frac_col = f"Component{i}_fraction"
            prop_col = f"Component{i}_Property{j}"
            inter_col = f"{frac_col}_x_{prop_col}"
            df[inter_col] = df[frac_col] * df[prop_col]
    return df

# Add interactions to training data
X_train_interact = X_train.copy()
X_val_interact = X_val.copy()
test_interact = test.copy()

X_train_interact = add_interaction_features(X_train_interact)
X_val_interact = add_interaction_features(X_val_interact)
test_interact = add_interaction_features(test_interact)

print(f"Tool Added {50} interaction features")  # 5 components × 10 properties
print(f"Data New training shape: {X_train_interact.shape}")

# Retrain models for hard targets with interactions
val_preds_exp3 = val_preds_exp2.copy()

for target in hard_targets:
    print(f"\nTool Retraining {target} with interaction features...")
    
    target_idx = target_cols.index(target)
    y_train_log = np.log1p(np.maximum(y_train[target], 1e-6))
    y_val_log = np.log1p(np.maximum(y_val[target], 1e-6))
    
    # Train with quantile regression objective
    train_set = lgb.Dataset(X_train_interact, label=y_train[target])  # No log transform
    val_set = lgb.Dataset(X_val_interact, label=y_val[target])
    
    params = {
        "objective": "quantile",  # Try quantile regression
        "alpha": 0.5,  # Median prediction
        "learning_rate": 0.02,
        "num_leaves": 128,
        "max_depth": 8,
        "reg_alpha": 1.0,
        "reg_lambda": 1.0,
        "n_estimators": 2000,
        "metric": "mape",
        "verbosity": -1,
        "seed": 42
    }
    
    model = lgb.train(
        params,
        train_set,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
    )
    
    # Make predictions
    val_preds_new = model.predict(X_val_interact)
    val_preds_exp3[:, target_idx] = val_preds_new
    improved_models[target] = model
    
    # Show results
    old_mape = mean_absolute_percentage_error(y_val[target], val_preds_exp2[:, target_idx])
    new_mape = mean_absolute_percentage_error(y_val[target], val_preds_new)
    improvement = (old_mape - new_mape) / old_mape * 100
    print(f"OK {target}: {old_mape:.6f} → {new_mape:.6f} ({improvement:+.1f}%)")

# Overall Experiment 3 MAPE
exp3_mape = mean_absolute_percentage_error(y_val, val_preds_exp3)
improvement = (exp2_mape - exp3_mape) / exp2_mape * 100
print(f"\nData Experiment 3 MAPE: {exp3_mape:.6f} ({improvement:+.1f}% vs Exp2)")

## Data Comparative Analysis

In [ ]:
# Compare all experiments
experiments = [
    ('Exp1: Baseline', exp1_mape),
    ('Exp2: Target Tuning', exp2_mape),
    ('Exp3: Interactions', exp3_mape)
]

print("Data Experiment Comparison")
print("=" * 50)
for name, mape in experiments:
    print(f"{name}: {mape:.6f}")

# Calculate improvements
baseline_mape = exp1_mape
best_mape = min(exp1_mape, exp2_mape, exp3_mape)
total_improvement = (baseline_mape - best_mape) / baseline_mape * 100

print(f"\nTarget Best MAPE: {best_mape:.6f}")
print(f"Chart Total improvement: {total_improvement:.1f}%")

# Visualize experiment progression
exp_names = [name.split(':')[0] for name, _ in experiments]
exp_mapes = [mape for _, mape in experiments]

plt.figure(figsize=(10, 6))
bars = plt.bar(exp_names, exp_mapes)
plt.title('MAPE Progression Across Experiments')
plt.ylabel('Mean Absolute Percentage Error')
plt.ylim(min(exp_mapes) * 0.95, max(exp_mapes) * 1.05)

# Add value labels on bars
for bar, mape in zip(bars, exp_mapes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.00001, 
             f'{mape:.6f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Target Final Model and Submission

In [ ]:
# Create final submission using best approach
print("Target Creating final submission...")

# Prepare test features
test_features = test[feature_cols].copy()
test_features_interact = add_interaction_features(test_features)

# Make predictions for all targets
test_preds = np.zeros((test.shape[0], len(target_cols)))

for i, target in enumerate(target_cols):
    if target in improved_models:
        # Use improved model with interactions
        test_preds[:, i] = improved_models[target].predict(test_features_interact)
    else:
        # Use baseline model
        test_preds[:, i] = models_exp1[i].predict(test_features)

# Create submission
submission = pd.DataFrame(test_preds, columns=target_cols)

# Round to appropriate precision
submission = submission.round(6)

# Save submission
submission.to_csv('iterative_submission.csv', index=False)

print("OK Submission saved as 'iterative_submission.csv'")
print(f"Data Submission shape: {submission.shape}")

# Preview
print("\nSearch Submission preview:")
print(submission.head())

## Chart Key Lessons and Insights

In [ ]:
print("Celebration Iterative Development Complete!")
print("=" * 60)
print("\nKey Key Lessons Learned:")
print("\n1. Target Target-Specific Optimization:")
print("   • Not all targets are equally difficult to predict")
print("   • Focus optimization efforts on challenging targets")
print("   • Log transformation can help with skewed targets")
print("\n2. Tool Feature Engineering Impact:")
print("   • Weighted blend features capture fundamental relationships")
print("   • Interaction features can improve performance for hard targets")
print("   • More features aren't always better - quality matters")
print("\n3. Lab Experimentation Strategy:")
print("   • Systematic testing of different approaches")
print("   • Build upon previous experiments")
print("   • Track improvements quantitatively")
print("\n4. Data Advanced Techniques:")
print("   • Quantile regression for robust predictions")
print("   • Early stopping prevents overfitting")
print("   • Ensemble approaches combine strengths")

print(f"\nChart Final Performance: {best_mape:.6f} MAPE")
print(f"Chart Total Improvement: {total_improvement:.1f}% from baseline")
print("\nLight Next Steps:")
print("• Physics-based features (linear blending rules)")
print("• Hyperparameter optimization (Optuna)")
print("• Cross-validation for robust evaluation")
print("• Ensemble methods with model stacking")